# Step 1 — Load Data

Localiza el dataset de petenfire v1 y lo expone en `data/raw/features.parquet` (via symlink).

**No carga las 136M filas en RAM.** Todo se hace con pyarrow a nivel de metadata y muestras pequeñas.

**Output**: `data/raw/features.parquet` → symlink a `data/processed/m1_dataset.parquet`

In [1]:
import sys
from pathlib import Path

# Agregar src/ al path para imports
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / 'src'))

from project_config import init_notebook
config = init_notebook()

import utils
logger = utils.init_logger('info')

📂 Project: petenfire2
📁 Root: /Users/mws/Documents/proyectos/AiFire/petenfire2
✅ Config loaded & imports configured


## 1.1 Localizar dataset de v1

In [2]:
project_folder = Path(config['project_folder'])

# Dataset fuente: m1_dataset.parquet de petenfire v1
v1_dataset = project_folder / 'data' / 'processed' / 'm1_dataset.parquet'

assert v1_dataset.exists(), (
    f"No existe el dataset de v1 en: {v1_dataset}\n"
    f"Verificar que el symlink 'data' apunta a ../petenfire/data"
)

logger.info('Dataset v1 encontrado: %s', v1_dataset)
print(f'Tamaño: {v1_dataset.stat().st_size / 1e9:.2f} GB')

2026-05-18 14:35:15,679 :: petenfire2 :: INFO :: Dataset v1 encontrado: /Users/mws/Documents/proyectos/AiFire/petenfire2/data/processed/m1_dataset.parquet


Tamaño: 3.19 GB


## 1.2 Crear symlink en data/raw/features.parquet

Usamos symlink para evitar copiar 3 GB. El resto del pipeline lee siempre desde `data/raw/features.parquet`.

In [3]:
raw_path = project_folder / config['data']['raw']['local_path']
raw_path.mkdir(parents=True, exist_ok=True)

out_path = raw_path / 'features.parquet'

if out_path.exists() or out_path.is_symlink():
    logger.info('features.parquet ya existe: %s', out_path)
else:
    out_path.symlink_to(v1_dataset.resolve())
    logger.info('Symlink creado: %s -> %s', out_path, v1_dataset)

print(f'Destino: {out_path}')
print(f'Es symlink: {out_path.is_symlink()}')
print(f'Resuelve a: {out_path.resolve()}')

2026-05-18 14:35:15,683 :: petenfire2 :: INFO :: features.parquet ya existe: /Users/mws/Documents/proyectos/AiFire/petenfire2/data/raw/features.parquet


Destino: /Users/mws/Documents/proyectos/AiFire/petenfire2/data/raw/features.parquet
Es symlink: True
Resuelve a: /Users/mws/Documents/proyectos/AiFire/petenfire/data/processed/m1_dataset.parquet


## 1.3 Validar schema y conteo de filas

In [4]:
import pyarrow.parquet as pq

pf = pq.ParquetFile(out_path)
schema = pf.schema_arrow

print(f'Filas totales  : {pf.metadata.num_rows:,}')
print(f'Row groups     : {pf.metadata.num_row_groups}')
print(f'Columnas       : {len(schema)}')
print()
print('Schema:')
for field in schema:
    print(f'  {field.name:<30} {str(field.type)}')

Filas totales  : 136,063,084
Row groups     : 137
Columnas       : 28

Schema:
  cell_id                        large_string
  date                           date32[day]
  fire_occurred                  bool
  T2M                            double
  RH2M                           double
  WS10M                          double
  PRECTOTCORR                    double
  fwi                            double
  ffmc_val                       double
  dmc_val                        double
  dc_val                         double
  isi_val                        double
  bui_val                        double
  prec_acc7d                     double
  prec_acc14d                    double
  elevation_m                    double
  slope_deg                      double
  aspect_deg                     double
  dist_roads_km                  double
  dist_settlements_km            double
  is_protected_area              bool
  month                          int32
  day_of_year                    in

## 1.4 Inspeccionar muestra sin cargar todo en RAM

Leemos un row group (≈ 1M filas) para ver distribuciones básicas.

In [5]:
import pandas as pd

# Leer primer row group (~1M filas) para inspección
sample = pf.read_row_group(0).to_pandas()

print(f'Muestra: {len(sample):,} filas')
print()
sample.head(3)

Muestra: 1,048,576 filas



,cell_id,date,fire_occurred,T2M,RH2M,WS10M,PRECTOTCORR,fwi,ffmc_val,dmc_val,...,dist_roads_km,dist_settlements_km,is_protected_area,month,day_of_year,year,split,ndvi,ndvi_lag7,ndvi_lag14
0,r0000_c0000,2018-01-01,False,22.68,87.35,1.30,1.53,0.234094,70.140882,1.462786,...,1.949526,9.159272,False,1,1,2018,train,0.602000,0.602,0.602
1,r0000_c0000,2018-01-02,False,20.77,90.78,2.37,4.22,0.023241,43.540787,0.248241,...,1.949526,9.159272,False,1,2,2018,train,0.594869,0.602,0.602
2,r0000_c0000,2018-01-03,False,20.89,86.36,2.37,0.04,0.170235,58.609937,1.369261,...,1.949526,9.159272,False,1,3,2018,train,0.587737,0.602,0.602


In [6]:
# Rango de fechas (sin cargar todo — leemos columna date por batch)
dates_min = []
dates_max = []

for batch in pf.iter_batches(batch_size=1_000_000, columns=['date']):
    col = batch.column('date')
    dates_min.append(col.to_pylist()[0])
    dates_max.append(col.to_pylist()[-1])

print(f'Rango de fechas: {min(dates_min)} → {max(dates_max)}')

Rango de fechas: 2018-01-01 → 2024-12-31


## 1.5 Distribución del target (fire_occurred)

In [7]:
# Contar positivos/negativos sin cargar todo — iteramos por batches
target_col = config['model']['objective_column']  # fire_occurred

n_positive = 0
n_total = 0

for batch in pf.iter_batches(batch_size=5_000_000, columns=[target_col]):
    col = batch.column(target_col).to_pylist()
    n_positive += sum(col)
    n_total += len(col)

n_negative = n_total - n_positive
ratio = n_negative / n_positive

print(f'Total filas     : {n_total:,}')
print(f'Positivos (fire): {n_positive:,} ({n_positive/n_total*100:.4f}%)')
print(f'Negativos       : {n_negative:,} ({n_negative/n_total*100:.4f}%)')
print(f'Ratio neg/pos   : {ratio:.1f}:1')
print()
print(f'scale_pos_weight sugerido: {ratio:.0f}')

Total filas     : 136,063,084
Positivos (fire): 87,659 (0.0644%)
Negativos       : 135,975,425 (99.9356%)
Ratio neg/pos   : 1551.2:1

scale_pos_weight sugerido: 1551


## 1.6 Distribución por año (split temporal)

In [8]:
# Leer columnas year + fire_occurred para ver distribución anual
# Leemos en chunks y acumulamos
import collections

year_counts = collections.defaultdict(lambda: {'total': 0, 'fires': 0})

for batch in pf.iter_batches(batch_size=5_000_000, columns=['year', target_col]):
    years = batch.column('year').to_pylist()
    fires = batch.column(target_col).to_pylist()
    for y, f in zip(years, fires):
        year_counts[y]['total'] += 1
        year_counts[y]['fires'] += int(f)

print(f"{'Año':<6} {'Filas':>12} {'Fuegos':>10} {'%':>8} {'Split':<8}")
print('-' * 50)
splits = config['data']['temporal_split']
for year in sorted(year_counts):
    d = year_counts[year]
    pct = d['fires'] / d['total'] * 100
    if year <= 2022:
        split = 'TRAIN'
    elif year == 2023:
        split = 'VAL'
    else:
        split = 'TEST'
    print(f"{year:<6} {d['total']:>12,} {d['fires']:>10,} {pct:>7.4f}% {split:<8}")

Año           Filas     Fuegos        % Split   
--------------------------------------------------
2018     19,422,380     10,268  0.0529% TRAIN   
2019     19,422,380     15,793  0.0813% TRAIN   
2020     19,475,592     11,721  0.0602% TRAIN   
2021     19,422,380      7,756  0.0399% TRAIN   
2022     19,422,380      8,624  0.0444% TRAIN   
2023     19,422,380     16,765  0.0863% VAL     
2024     19,475,592     16,732  0.0859% TEST    


## 1.7 Verificación de columnas esperadas

In [9]:
expected_columns = [
    # Identificadores
    'cell_id', 'date', 'year',
    # Target
    'fire_occurred',
    # Clima
    'T2M', 'RH2M', 'WS10M', 'PRECTOTCORR', 'prec_acc7d', 'prec_acc14d',
    # FWI
    'fwi', 'ffmc_val', 'dmc_val', 'dc_val', 'isi_val', 'bui_val',
    # Topografía y estáticas
    'elevation_m', 'slope_deg', 'aspect_deg',
    'dist_roads_km', 'dist_settlements_km', 'is_protected_area',
    # NDVI
    'ndvi', 'ndvi_lag7', 'ndvi_lag14',
    # Temporales
    'month', 'day_of_year',
]

available = set(schema.names)
missing = [c for c in expected_columns if c not in available]
extra = [c for c in available if c not in expected_columns]

print(f'Columnas esperadas: {len(expected_columns)}')
print(f'Columnas en dataset: {len(available)}')
print()

if missing:
    print(f'FALTANTES: {missing}')
else:
    print('Todas las columnas esperadas presentes.')

if extra:
    print(f'Extra (no esperadas): {extra}')

# Notar columnas que se agregarán en Step 5
print()
print('Columnas pendientes de construir en Step 5 (feature engineering):')
print('  fire_lag_1d, fire_lag_3d, fire_lag_7d, fire_lag_14d')
print('  fire_neighbors_3x3_7d, fire_neighbors_5x5_7d')

Columnas esperadas: 27
Columnas en dataset: 28

Todas las columnas esperadas presentes.
Extra (no esperadas): ['split']

Columnas pendientes de construir en Step 5 (feature engineering):
  fire_lag_1d, fire_lag_3d, fire_lag_7d, fire_lag_14d
  fire_neighbors_3x3_7d, fire_neighbors_5x5_7d


## Resumen

- Dataset de v1 localizado y expuesto en `data/raw/features.parquet`
- 136M filas, 28 columnas, rango 2018-2024
- Target `fire_occurred`: ~0.03% de positivos → desbalance extremo
- Split temporal: TRAIN 2018-2022 / VAL 2023 / TEST 2024
- Las fire_lag y spatial features se construyen en **Step 5**

**Siguiente paso**: `step2_exploration.ipynb`